In [64]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import col,udf
import time
from pyspark.sql.types import StringType,IntegerType
spark = SparkSession.builder.appName("AdvanceSparkApp").getOrCreate()

# Create DataFrame
df = spark.range(0, 1000000).withColumn("value", col("id") % 1000)



In [28]:
# Check initial partitions
print("Before Partitions:", df.rdd.getNumPartitions())

Before Partitions: 2


In [29]:
# Increase partitions
df_repartitioned = df.repartition(10)
print("After Partitions:", df_repartitioned.rdd.getNumPartitions())

[Stage 16:>                                                         (0 + 2) / 2]

After Partitions: 10


[Stage 16:=============================>                            (1 + 1) / 2]

In [30]:
# Decrease partitions
df_coalesced = df_repartitioned.coalesce(2)
print("After Coalesced:", df_coalesced.rdd.getNumPartitions())

After Coalesced: 2


[Stage 17:>                                                         (0 + 2) / 2]

In [31]:
df.write.mode("overwrite").csv("output/DATATable", header=True)

# ==================or====================

  # df.write \
  # .mode("overwrite") \
  # .option("header", True) \
  # .csv("output/DATATable")

In [32]:
df2 = spark.read.csv(
    "output/DATATable",
    header=True,
    inferSchema=True
)

df2.show(5)

+------+-----+
|    id|value|
+------+-----+
|500000|    0|
|500001|    1|
|500002|    2|
|500003|    3|
|500004|    4|
+------+-----+
only showing top 5 rows



In [33]:
df2.count()

1000000

In [34]:
#optimization ands caching
optimized_df = df.filter(col("value")>500).filter(col("id")<500000).select("id","value")

In [35]:
optimized_df.show(2)

+---+-----+
| id|value|
+---+-----+
|501|  501|
|502|  502|
+---+-----+
only showing top 2 rows



In [36]:
df.explain()

== Physical Plan ==
*(1) Project [id#83L, (id#83L % 1000) AS value#85L]
+- *(1) Range (0, 1000000, step=1, splits=2)




In [49]:
start_time = time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1. optimized execution | count : {count_uncached} | time: {round(end_time-start_time,4)} seconds")

1. optimized execution | count : 249500 | time: 0.2757 seconds


In [50]:
optimized_df.cache()

26/06/13 06:07:00 WARN CacheManager: Asked to cache already cached data.


DataFrame[id: bigint, value: bigint]

In [51]:
start_time = time.time()
count_cached = optimized_df.count()
end_time = time.time()
print(f"1. optimized execution | count : {count_cached} | time: {round(end_time-start_time,4)} seconds")

1. optimized execution | count : 249500 | time: 0.1381 seconds


In [52]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [69]:
data = [("Alice", 25), ("Bob", 17), ("Charlie", 35), ("David", 15)]
df1 = spark.createDataFrame(data, ["Name", "Age"])

In [70]:
def categorize_age(age):
   if age>=18:
    return "adult"
   else:
       return "minor"

In [71]:
age_categorize_udf= udf(categorize_age,StringType())

In [73]:
df_method1 = df1.withColumn("Category",age_categorize_udf(col("Age")))
print("Method 1: Standard udf via dataframe api")
df_method1.show()

Method 1: Standard udf via dataframe api


[Stage 60:>                                                         (0 + 1) / 1]

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   adult|
|    Bob| 17|   minor|
|Charlie| 35|   adult|
|  David| 15|   minor|
+-------+---+--------+



In [74]:
spark.udf.register("sql_categorize_age", categorize_age, StringType())

df1.createOrReplaceTempView("people")

In [75]:
sql_df = spark.sql("""
    SELECT
       Name,
       Age, 
       sql_categorize_age(Age) AS Category
   FROM people
""")

sql_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   adult|
|    Bob| 17|   minor|
|Charlie| 35|   adult|
|  David| 15|   minor|
+-------+---+--------+



In [78]:
def can_drive(Category):
   if "adult":
    return "yes"
   else:
       return "no"

In [79]:
spark.udf.register("sql_can_drive", can_drive, StringType())

df1.createOrReplaceTempView("people")

26/06/13 07:07:32 WARN SimpleFunctionRegistry: The function sql_can_drive replaced a previously registered function.


In [81]:
sql_df = spark.sql("""
    SELECT
       Name,
       Age,
       Category,
       sql_can_drive(Category) as can_drive 
   FROM people
""")

sql_df.show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `Category` cannot be resolved. Did you mean one of the following? [`Age`, `Name`].; line 5 pos 7;
'Project [Name#740, Age#741L, 'Category, 'sql_can_drive('Category) AS can_drive#785]
+- SubqueryAlias people
   +- View (`people`, [Name#740,Age#741L])
      +- LogicalRDD [Name#740, Age#741L], false
